# DNA Data Preprocessing

This notebook creates cleaned, model-ready datasets for `train` and `test`:

1. **Sequence one-hot encoding** for nucleotide strings (`A`, `C`, `G`, `T`).
2. **Engineered numeric features**:
   - sequence length
   - `%A`, `%T`, `%C`, `%G`
   - first 50 bases composition: `[pA, pT, pC, pG]`
   - last 50 bases composition: `[pA, pT, pC, pG]`
3. **Target one-hot encoding** for `GeneType`.

Outputs are saved under `clean_data/train/` and `clean_data/test/` with consistent row indexing per split.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
CLEAN_DIR = BASE_DIR / 'clean_data'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train': DATA_DIR / 'actual_train_dataset.csv',
    'test': DATA_DIR / 'actual_test_dataset.csv',
}

SPLIT_OUTPUT_DIRS = {split_name: CLEAN_DIR / split_name for split_name in SPLITS}
for out_dir in SPLIT_OUTPUT_DIRS.values():
    out_dir.mkdir(parents=True, exist_ok=True)

NUCLEOTIDE_COL = 'NucleotideSequence'
TARGET_COL = 'GeneType'
INDEX_COL = 'row_index'

In [ ]:
BASE_TO_INDEX = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
ORDERED_BASES = ['A', 'T', 'C', 'G']


def clean_sequence(seq: str) -> str:
    """Normalize sequence and keep only A/C/G/T."""
    if pd.isna(seq):
        return ''
    seq = str(seq).strip().upper().replace('<', '').replace('>', '')
    return ''.join(ch for ch in seq if ch in BASE_TO_INDEX)


def encode_sequence_one_hot(seq: str) -> np.ndarray:
    """Encode a sequence into shape (L, 4) with A,C,G,T channel order."""
    encoded = np.zeros((len(seq), 4), dtype=np.uint8)
    for i, base in enumerate(seq):
        encoded[i, BASE_TO_INDEX[base]] = 1
    return encoded


def base_percentages(seq: str):
    """Return %A, %T, %C, %G for a sequence."""
    length = len(seq)
    if length == 0:
        return [0.0, 0.0, 0.0, 0.0]
    return [seq.count(b) / length for b in ORDERED_BASES]


def edge_composition(seq: str, window: int = 50):
    """Return first_50 and last_50 base percentages in A,T,C,G order."""
    first = seq[:window]
    last = seq[-window:] if len(seq) >= window else seq
    return base_percentages(first), base_percentages(last)


def build_numeric_features(seq: str) -> dict:
    """Build handcrafted numeric feature dictionary for one sequence."""
    length = len(seq)
    pct_a, pct_t, pct_c, pct_g = base_percentages(seq)
    first_50, last_50 = edge_composition(seq, window=50)

    return {
        'length': float(length),
        'pct_A': pct_a,
        'pct_T': pct_t,
        'pct_C': pct_c,
        'pct_G': pct_g,
        'first_50_pA': first_50[0],
        'first_50_pT': first_50[1],
        'first_50_pC': first_50[2],
        'first_50_pG': first_50[3],
        'last_50_pA': last_50[0],
        'last_50_pT': last_50[1],
        'last_50_pC': last_50[2],
        'last_50_pG': last_50[3],
    }

In [ ]:
# Build a global, stable gene-type index so one-hot columns stay consistent
# across train/test.
all_gene_types = []
for split_name, csv_path in SPLITS.items():
    df_tmp = pd.read_csv(csv_path)
    all_gene_types.extend(df_tmp[TARGET_COL].dropna().astype(str).tolist())

unique_gene_types = sorted(set(all_gene_types))
gene_to_idx = {gene: i for i, gene in enumerate(unique_gene_types)}
num_classes = len(unique_gene_types)

mapping_df = pd.DataFrame({
    'gene_type': unique_gene_types,
    'class_index': list(range(num_classes)),
})
mapping_df.to_csv(CLEAN_DIR / 'gene_type_mapping.csv', index=False)

print(f'Found {num_classes} unique GeneType classes.')
print('Saved class mapping to clean_data/gene_type_mapping.csv')

In [ ]:
split_summaries = []

for split_name, csv_path in SPLITS.items():
    df = pd.read_csv(csv_path)
    split_out_dir = SPLIT_OUTPUT_DIRS[split_name]

    # Preserve CSV row identity in a dedicated column for alignment checks.
    if 'Unnamed: 0' in df.columns:
        df = df.rename(columns={'Unnamed: 0': INDEX_COL})
    elif INDEX_COL not in df.columns:
        df[INDEX_COL] = np.arange(len(df), dtype=np.int64)

    # Clean sequence text
    df[NUCLEOTIDE_COL] = df[NUCLEOTIDE_COL].apply(clean_sequence)

    # 1) Sequence one-hot encodings (variable-length arrays)
    sequence_onehot = [encode_sequence_one_hot(seq) for seq in df[NUCLEOTIDE_COL]]
    np.save(split_out_dir / 'sequence_onehot.npy', np.array(sequence_onehot, dtype=object), allow_pickle=True)

    # 2) Numeric handcrafted feature table
    features_df = pd.DataFrame([build_numeric_features(seq) for seq in df[NUCLEOTIDE_COL]])
    features_df.insert(0, INDEX_COL, df[INDEX_COL].to_numpy())
    features_df.to_csv(split_out_dir / 'features.csv', index=False)

    # 3) Target one-hot vectors
    y_indices = df[TARGET_COL].astype(str).map(gene_to_idx).to_numpy()
    y_onehot = np.zeros((len(df), num_classes), dtype=np.uint8)
    y_onehot[np.arange(len(df)), y_indices] = 1
    np.save(split_out_dir / 'y_onehot.npy', y_onehot)

    targets_df = pd.DataFrame({
        INDEX_COL: df[INDEX_COL].to_numpy(),
        TARGET_COL: df[TARGET_COL].astype(str).to_numpy(),
        'class_index': y_indices,
    })
    targets_df.to_csv(split_out_dir / 'targets.csv', index=False)

    # Alignment/meta file to keep indexing explicit and reproducible
    alignment_df = pd.DataFrame({
        INDEX_COL: df[INDEX_COL].to_numpy(),
        'sequence_length': df[NUCLEOTIDE_COL].str.len().to_numpy(),
    })
    alignment_df.to_csv(split_out_dir / 'index_alignment.csv', index=False)

    split_summaries.append({
        'split': split_name,
        'rows': len(df),
        'avg_length': float(df[NUCLEOTIDE_COL].str.len().mean()),
    })

summary_df = pd.DataFrame(split_summaries)
summary_df.to_csv(CLEAN_DIR / 'dataset_summary.csv', index=False)

print('Saved processed data to clean_data/:')
for _, row in summary_df.iterrows():
    print(f"- {row['split']}: rows={int(row['rows'])}, avg_length={row['avg_length']:.2f}")

In [ ]:
# Quick validation checks for indexing consistency
for split_name in SPLITS:
    split_out_dir = SPLIT_OUTPUT_DIRS[split_name]
    f_df = pd.read_csv(split_out_dir / 'features.csv')
    t_df = pd.read_csv(split_out_dir / 'targets.csv')
    i_df = pd.read_csv(split_out_dir / 'index_alignment.csv')

    same_index = f_df[INDEX_COL].equals(t_df[INDEX_COL]) and f_df[INDEX_COL].equals(i_df[INDEX_COL])
    print(f'{split_name}: index consistent -> {same_index}')